# 第104章 混淆矩阵与分类指标

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 19 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 多分类与Softmax  →  **本章任务：** 混淆矩阵与分类指标  →  **下一步：** ROC、PR曲线与决策阈值
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

拿到模型给出的"预测对/预测错"结果，光靠一条"准确率"根本说不清它好在哪——比如 100 单里漏掉了 90 单高风险交易，准确率却可能依然很高，损失的却是实打实的钱。


## 本章目标

学完本章，你将能够：

- **理解**：理解「混淆矩阵与分类指标」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「混淆矩阵与分类指标」的关键输出指标。
- **迁移**：能把「混淆矩阵与分类指标」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：拿到模型给出的"预测对/预测错"结果，光靠一条"准确率"根本说不清它好在哪——比如 100 单里漏掉了 90 单高风险交易，准确率却可能依然很高，损失的却是实打实的钱。混淆矩阵把预测与实际结果交叉成四个格子，再派生出精确率、召回率和 F1，帮你分清"该抓的有没有抓住"和"抓到的有多少是误伤"。做分类问题，无论贷款风控还是疾病筛查，都得靠它回答"该信哪些数字"。

- Precision=TP/(TP+FP)
- Recall=TP/(TP+FN)
- F1=2PR/(P+R)
- 指标选择取决于漏判与误报成本（打个比方：像收网——网眼太大漏掉值钱的鱼，太小捞上一堆没用的；到底要抓多少、容忍多少误伤，取决于你更心疼哪种。）


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `model.predict()`、`.fit()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 未确认正类定义 |
| 模型、公式与诊断 | `.ravel()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 类别不平衡时只看准确率 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-104 -->
### 数学推导｜混淆矩阵派生分类指标

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜把预测与真实标签交叉计数。** 正类预测中包含 $TP+FP$，真实正类中包含 $TP+FN$。

**第 2 步｜形成两个不同条件比例。** Precision 是“预测为正时有多准”，Recall 是“真实为正时找回多少”。

**第 3 步｜用调和平均合并。** 

$$
F_1=\frac{2}{1/P+1/R}=\frac{2PR}{P+R}
$$

调和平均会被较小的一项明显拉低，所以只有 precision 与 recall 都不差时 F1 才高。

**把上面的关系收束为本章计算式：**

$$
Precision=\frac{TP}{TP+FP},\qquad Recall=\frac{TP}{TP+FN},\qquad F_1=\frac{2PR}{P+R}
$$

**符号解释：** TP、FP、FN 分别是真阳性、假阳性、假阴性。

**代码对应：** 先明确哪个类别是正类，再通过混淆矩阵核对指标分子和分母。

**使用边界：** F1 隐含 precision 与 recall 同等重要，未必符合真实业务成本。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=93
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(
    X_train, y_train
)
pred = model.predict(X_test)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：调整模型参数，观察混淆矩阵与指标的变化

`96.3 示例 1` 用默认参数训练逻辑回归并得到测试集预测 `pred`。请把 `LogisticRegression` 的 `class_weight` 改为 `'balanced'` 重新训练，再重新计算测试集上的混淆矩阵与精确率、召回率，观察当模型开始对少数类样本加权后，TN/FP/FN/TP 四个格子发生了什么变化。


In [ ]:
try:
    # 请在下方填写代码
    from sklearn.metrics import confusion_matrix, precision_score, recall_score

    # 与 96.3 示例 1 相同的切分和管线，唯一改动：把 class_weight 换成 'balanced'
    # TODO: 把 _X_ 改成 'balanced'，让模型在训练时对少数类样本加权。

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print({"TN": tn, "FP": fp, "FN": fn, "TP": tp})
print("手算:", round(precision, 3), round(recall, 3), round(f1, 3))
print(
    "sklearn:",
    round(precision_score(y_test, pred), 3),
    round(recall_score(y_test, pred), 3),
    round(f1_score(y_test, pred), 3),
)


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X_reg = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y_reg = np.array([12, 15, 19, 23, 27, 31])
reg_baseline = DummyRegressor(strategy="mean").fit(X_reg, y_reg)
reg_model = LinearRegression().fit(X_reg, y_reg)
print("基线预测：", np.round(reg_baseline.predict(X_reg[:2]), 2))
print("模型预测：", np.round(reg_model.predict(X_reg[:2]), 2))
print(
    "基线MAE：",
    round(mean_absolute_error(y_reg, reg_baseline.predict(X_reg)), 2),
)
print(
    "模型MAE：", round(mean_absolute_error(y_reg, reg_model.predict(X_reg)), 2)
)


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_reg_changed = X_reg.copy()
X_reg_changed["visits"] = X_reg_changed["visits"] + 1
reg_changed_prediction = reg_model.predict(X_reg_changed)
print("原始前2个预测：", np.round(reg_model.predict(X_reg[:2]), 2))
print("访问次数+1后的预测：", np.round(reg_changed_prediction[:2], 2))
print(
    "预测变化：",
    np.round(reg_changed_prediction[:2] - reg_model.predict(X_reg[:2]), 2),
)


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 未确认正类定义
- 类别不平衡时只看准确率
- 把精确率和召回率混淆
- 没有把指标转换成实际错误数量


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 104.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 104.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 104.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

从混淆矩阵手算准确率、精确率、召回率和 F1，建立指标与错误成本的联系。


### 你已经掌握

- 识别 TN、FP、FN、TP
- 手算 Precision、Recall 与 F1
- 使用 classification_report
- 根据业务错误成本选择指标


### 需要注意

- 未确认正类定义
- 类别不平衡时只看准确率
- 把精确率和召回率混淆
- 没有把指标转换成实际错误数量


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# 完整答案：_X_ = 'balanced'
balanced_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
).fit(X_train, y_train)

pred_bal = balanced_model.predict(X_test)
tn2, fp2, fn2, tp2 = confusion_matrix(y_test, pred_bal).ravel()
precision2 = tp2 / (tp2 + fp2)
recall2 = tp2 / (tp2 + fn2)
print({"TN": tn2, "FP": fp2, "FN": fn2, "TP": tp2})
print("balanced 精确率 / 召回率:", round(precision2, 3), round(recall2, 3))


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_matrix = confusion_matrix(y_test, pred)
practice_total = practice_matrix.sum()
practice_accuracy = (
    practice_matrix[0, 0] + practice_matrix[1, 1]
) / practice_total
print("accuracy:", round(practice_accuracy, 3))
